# Generatyvinis DI: bendras LLM prieš 1 užduoties klasifikatorių su 20 Fakeddit antraščių

Nedidelis generatyvinio DI užduoties bandymas. 20 antraščių iš Fakeddit testo duomenų (10 netikrų, 10 tikrų) duodama bendram LLM (Google Gemini 3.5 Flash-Lite per API) su viena fiksuota užklausa: netikra ar tikra ir kodėl. Jo atsakymai palyginami su tikromis žymėmis ir su logistinės regresijos modeliu iš `01_ml_model.ipynb`.

Tai A būdas iš ataskaitos skyriaus (bendras LLM, tik užklausa, be šaltinių). 20 įrašų per mažai tiksliam tikslumui nustatyti; tikslas - pamatyti, kokius atsakymus ir paaiškinimus duoda kiekvienas būdas.

In [1]:
import json, time, requests, joblib, numpy as np, pandas as pd
from pathlib import Path

DATA = Path("../data/multimodal_only_samples-20261006T085257Z-1-001/multimodal_only_samples")
MODEL_PATH = Path("../models/logreg_tfidf.joblib")
CACHE = Path("../models/gemini_20_titles.csv")  # LLM atsakymus saugom, kad kiekvienas paleidimas nekainuotu ir neduotu kitu atsakymu
FEATURES = ["clean_title", "score", "num_comments", "upvote_ratio"]
SUBTYPES = {0: "true", 1: "satire/parody", 2: "false connection", 3: "imposter content",
            4: "manipulated content", 5: "misleading content"}
LLM = "gemini-3.5-flash-lite"  # nemokamas planas: Flash leidzia tik 20 uzklausu per diena, Lite turi atskira kvota
SEED = 42


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## 1. 20 testo antraščių atranka
Atsitiktinai 10 netikrų ir 10 tikrų. Įrašai be įsitraukimo skaičių praleidžiami: 7 užduotyje paaiškėjo, kad jie yra trumpas kelias į atsakymą (visi netikri), todėl klasifikatorius atrodytų geresnis, nei yra.

In [2]:
test = pd.read_csv(DATA / "multimodal_test_public.tsv", sep="	", usecols=FEATURES + ["2_way_label", "6_way_label", "subreddit"])
test = test.dropna(subset=["num_comments"])  # be skaiciu irasai visi netikri, modeliui tai per lengva
sample = pd.concat([test[test["2_way_label"] == k].sample(10, random_state=SEED) for k in (0, 1)]).sample(frac=1, random_state=SEED).reset_index(drop=True)  # sumaisom, kad netikros ir tikros nebutu is eiles
sample["label"] = sample["2_way_label"].map({0: "fake", 1: "real"})
sample[["clean_title", "subreddit", "label"]]

,clean_title,subreddit,label
0,the angel gabriel visits joseph bc,fakehistoryporn,fake
1,the flying motorcycle is escaping,photoshopbattles,real
2,bending it,photoshopbattles,real
3,he was surprised to see me,pareidolia,fake
4,it will take hours to get home,pareidolia,fake
5,why is this dog wearing a watch oh,pareidolia,fake
6,this tube tells you where to press to dispense,mildlyinteresting,real
7,my penis,misleadingthumbnails,fake
8,a halfway repaved road,mildlyinteresting,real
9,breast cancer drug that can extend lives appro...,upliftingnews,real


## 2. 1 užduoties klasifikatorius
Išsaugota grandinė, be pakeitimų. Ji mato antraštę ir tris įsitraukimo skaičius.

In [3]:
model = joblib.load(MODEL_PATH)
sample["clf_p_real"] = model.predict_proba(sample[FEATURES])[:, 1]
sample["classifier"] = np.where(sample["clf_p_real"] >= 0.5, "real", "fake")

C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from versi

C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.9.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Pirik\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 3. Gemini, tik užklausa
Užklausa fiksuota, o temperatūra 0, todėl atsakymai tiek pastovūs, kiek leidžia API. Modelis turi atsakyti JSON formatu (`label`, `reason`), todėl atsakymus lengva palyginti. Siunčiama tik antraštė, be įsitraukimo skaičių ir be paveikslėlio.

API raktas skaitomas iš git ignoruojamo `.env` failo (`GEMINI_API_KEY=...`). Atsakymai po kiekvienos užklausos išsaugomi `models/gemini_20_titles.csv`; norint klausti iš naujo, failą reikia ištrinti. Nemokamas API planas leidžia tik kelias užklausas per minutę, todėl daromos pauzės.

In [4]:
PROMPT = """You help a news fact-checking team. Below is the title of a Reddit post that was shared with an image (the image is not shown).
Decide if the post is FAKE (satire or parody, misleading, manipulated or photoshopped, false connection between title and image, imposter) or REAL (a true, accurately described photo or news item).
Answer with the label and one short sentence explaining why.

Title: {title}"""

SCHEMA = {"type": "OBJECT", "properties": {"label": {"type": "STRING", "enum": ["fake", "real"]}, "reason": {"type": "STRING"}},
          "required": ["label", "reason"]}

def ask_llm(title):
    key = dict(l.split("=", 1) for l in Path("../.env").read_text().split()).get("GEMINI_API_KEY")  # rakto nerasom i koda, jis tik .env faile
    body = {"contents": [{"parts": [{"text": PROMPT.format(title=title)}]}],
            "generationConfig": {"temperature": 0, "responseMimeType": "application/json", "responseSchema": SCHEMA,
                                 "thinkingConfig": {"thinkingLevel": "low"}}}  # trumpas "galvojimas": atsakymas per kelias sekundes, ne minutes
    for attempt in range(5):
        r = requests.post(f"https://generativelanguage.googleapis.com/v1beta/models/{LLM}:generateContent",
                          headers={"x-goog-api-key": key}, json=body, timeout=120)
        if r.status_code not in (429, 503):
            break
        time.sleep(60)  # 429/503 reiskia per daug uzklausu ar perkrauta, palaukiam ir bandom dar
    r.raise_for_status()
    return json.loads(r.json()["candidates"][0]["content"]["parts"][0]["text"])

answers = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame(columns=["clean_title", "label", "reason"])
for t in sample["clean_title"]:
    if t in set(answers["clean_title"]):
        continue  # jau paklausta anksciau
    answers = pd.concat([answers, pd.DataFrame([{"clean_title": t, **ask_llm(t)}])], ignore_index=True)
    answers.to_csv(CACHE, index=False)  # saugom po kiekvieno atsakymo, kad nutrukus nereiktu klausti is naujo
    time.sleep(13)  # nemokamas planas leidzia tik kelias uzklausas per minute
sample = sample.merge(answers.rename(columns={"label": "llm", "reason": "llm_reason"}), on="clean_title")

## 4. Palyginimas

In [5]:
pd.set_option("display.max_colwidth", 120)
sample["subtype"] = sample["6_way_label"].map(SUBTYPES)
sample[["clean_title", "subtype", "label", "classifier", "llm", "llm_reason"]]

,clean_title,subtype,label,classifier,llm,llm_reason
0,the angel gabriel visits joseph bc,false connection,fake,fake,fake,"The title is humorous and refers to a religious figure, which is characteristic of a meme or satirical post rather t..."
1,the flying motorcycle is escaping,true,real,real,fake,"The title makes an absurd claim about a flying motorcycle escaping, which is indicative of satire, fiction, or a mis..."
2,bending it,true,real,fake,fake,The title 'bending it' is vague and lacks sufficient verifiable context to confirm the authenticity of the accompany...
3,he was surprised to see me,false connection,fake,fake,real,"The title is a common, subjective caption for a casual photo of an unexpected encounter and does not present verifia..."
4,it will take hours to get home,false connection,fake,fake,real,"The title expresses a common, verifiable sentiment about traffic conditions without making any false factual claims."
5,why is this dog wearing a watch oh,false connection,fake,real,real,"The title appears to be a casual, genuine observation of a dog wearing a watch."
6,this tube tells you where to press to dispense,true,real,real,real,This describes a common feature found on some toothpaste or cream tubes that indicate where to apply pressure for op...
7,my penis,false connection,fake,fake,fake,The title is nonsensical and does not correspond to a credible news item or verified factual photograph.
8,a halfway repaved road,true,real,real,real,This is a plausible everyday occurrence that does not exhibit indicators of digital manipulation or satire.
9,breast cancer drug that can extend lives approved for the nhs,true,real,real,real,This headline reflects legitimate news about pharmaceutical approvals by the UK's National Health Service.


In [6]:
pd.DataFrame({name: {"correct (of 20)": (sample[col] == sample["label"]).sum(),
                     "fake found (of 10)": ((sample[col] == "fake") & (sample["label"] == "fake")).sum(),
                     "real found (of 10)": ((sample[col] == "real") & (sample["label"] == "real")).sum()}
              for name, col in [("Task 1 classifier", "classifier"), (f"LLM ({LLM})", "llm")]})

,Task 1 classifier,LLM (gemini-3.5-flash-lite)
correct (of 20),14,12
fake found (of 10),7,5
real found (of 10),7,7


In [7]:
pd.crosstab(sample["classifier"] == sample["label"], sample["llm"] == sample["label"]).rename_axis(index="classifier right", columns="LLM right")  # kur abu klysta, o kur tik vienas

LLM right,False,True
classifier right,,
False,2,4
True,6,8
